# Capítulo 8 — De um agente para muitos (multiagentes)

**Livro:** começa na página **260** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py: modelos via OpenRouter, com controle de gasto
from ia import *
import json, re, time, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO)

## How Many Agents Do I Need? · PDF p. 260

#### 🧪 Exercício: Supervisor que roteia

Crie dois "especialistas" (funções com *system prompts* diferentes: vendas e suporte técnico) e um supervisor que pede ao modelo para escolher o especialista (responda só `vendas` ou `suporte`) e repassa a pergunta.

In [ ]:
ESPECIALISTAS = {"vendas": "Você é vendedor: fale de preços, promoções e produtos. Seja breve.",
                 "suporte": "Você é do suporte técnico: ajude com defeitos e configuração. Seja breve."}
def supervisor(q):
    destino = perguntar(f"Quem deve responder: vendas ou suporte? Só a palavra.\nPergunta: {q}", temperature=0).strip().lower().strip(".")
    destino = destino if destino in ESPECIALISTAS else "suporte"
    return destino, perguntar(q, sistema=ESPECIALISTAS[destino])
for q in ["Tem desconto no combo teclado + mouse?", "Meu mouse não conecta no bluetooth."]:
    print(supervisor(q))

## Multiagent Coordination · PDF p. 287

#### 🧪 Exercício: Escritor e revisor

Coordene dois agentes: o **escritor** redige a descrição de um produto; o **revisor** devolve `APROVADO` ou uma crítica. Repita até aprovar (no máximo 3 rodadas).

In [ ]:
texto = perguntar("Escreva uma descrição de 2 frases para uma cadeira ergonômica, para um site de vendas.")
for rodada in range(3):
    critica = perguntar(f"Revise o texto. Se estiver claro, sem exageros e com no máximo 2 frases, responda só APROVADO. Senão, dê uma crítica curta.\n\n{texto}", modelo=MODELO_FORTE, temperature=0)
    print(f"rodada {rodada + 1}:", critica[:80])
    if critica.strip().upper().startswith("APROVADO"): break
    texto = perguntar(f"Reescreva o texto seguindo a crítica.\nTexto: {texto}\nCrítica: {critica}")
print(texto)

## Message Brokers and Event Buses · PDF p. 305

#### 🧪 Exercício: Barramento de eventos em memória

Implemente um `Barramento` com `assinar(topico, funcao)` e `publicar(topico, mensagem)`. Faça um agente de pedidos publicar `pedido.criado` e dois assinantes reagirem (estoque e e-mail).

In [ ]:
from collections import defaultdict
class Barramento:
    def __init__(self): self.assinantes = defaultdict(list)
    def assinar(self, topico, f): self.assinantes[topico].append(f)
    def publicar(self, topico, msg):
        for f in self.assinantes[topico]: f(msg)
bus = Barramento()
bus.assinar("pedido.criado", lambda m: print("📦 estoque reservou", m["itens"]))
bus.assinar("pedido.criado", lambda m: print("✉️ e-mail enviado para", m["cliente"]))
bus.publicar("pedido.criado", {"id": "A102", "cliente": "ana@exemplo.com", "itens": ["mouse"]})

## Managing State and Persistence · PDF p. 318

#### 🧪 Exercício: Salvar e retomar o estado do agente

Um agente executa 5 etapas de um processo. Salve o estado (etapa atual e resultados) num JSON após cada etapa. Simule uma queda na etapa 3 e mostre que, ao reiniciar, ele continua da etapa 3, sem refazer as anteriores.

In [ ]:
import os, tempfile
arq = os.path.join(tempfile.gettempdir(), "estado_agente.json")
if os.path.exists(arq): os.remove(arq)
def rodar(cair_em=None):
    estado = json.load(open(arq)) if os.path.exists(arq) else {"etapa": 1, "resultados": []}
    while estado["etapa"] <= 5:
        if estado["etapa"] == cair_em: print("💥 caiu na etapa", cair_em); return
        estado["resultados"].append(f"etapa {estado['etapa']} ok"); estado["etapa"] += 1
        json.dump(estado, open(arq, "w"))
    print(estado["resultados"])
rodar(cair_em=3); rodar(); os.remove(arq)